# Day 26 Tutorial：一轮主动学习池模拟

> **课程附带的 ESOL 池模拟。** “揭示标签”只模拟已有标签返回；没有开展真实合成、测试或下游任务推荐。

## Goal

在固定初始已标注集与预算下，执行一次“分歧选择—标签揭示—更新”，并和同预算随机选择比较。

## Setup

原 train 被确定性拆成初始已标注集和候选池。ESOL valid 不参与采集，只作同轮固定对照；它在课程中已被查看过，不是严格未见 test。模拟 oracle 到 query 固定后才接收池标签。

In [1]:
from pathlib import Path
import contextlib
import io
import warnings

from rdkit import RDLogger
RDLogger.DisableLog("rdApp.warning")

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    import deepchem as dc

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
np.random.seed(SEED)

def find_repo_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "public" / "esol.md").exists():
            return candidate
    raise RuntimeError("请从 ML-Learning 仓库内运行本教程。")

REPO_ROOT = find_repo_root()
CACHE_DIR = REPO_ROOT / ".cache" / "deepchem"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

featurizer = dc.feat.CircularFingerprint(size=1024, radius=2)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    tasks, datasets, transformers = dc.molnet.load_delaney(
        featurizer=featurizer,
        splitter="scaffold",
        transformers=[],
        reload=True,
        data_dir=str(CACHE_DIR),
        save_dir=str(CACHE_DIR),
    )

train_dataset, valid_dataset, _sealed_test_dataset = datasets
X_train = np.asarray(train_dataset.X)
y_train = np.asarray(train_dataset.y).reshape(-1)
X_valid = np.asarray(valid_dataset.X)
y_valid = np.asarray(valid_dataset.y).reshape(-1)
train_ids = np.asarray(train_dataset.ids).astype(str)
valid_ids = np.asarray(valid_dataset.ids).astype(str)

assert transformers == []
assert X_train.shape == (902, 1024) and y_train.shape == (902,)
assert X_valid.shape == (113, 1024) and y_valid.shape == (113,)
print("Task:", tasks[0])
print("Train / valid:", X_train.shape, X_valid.shape)
print("测试集对象保持封存，本教程不创建测试预测。")

Task: measured log solubility in mols per litre
Train / valid: (902, 1024) (113, 1024)
测试集对象保持封存，本教程不创建测试预测。


In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

INITIAL_SIZE = 160
BATCH_SIZE = 10
MEMBER_SEEDS = [11, 22, 33, 44, 55]
RANDOM_QUERY_SEED = 2026
rng = np.random.default_rng(SEED)
permutation = rng.permutation(len(X_train))
initial_idx = permutation[:INITIAL_SIZE]
pool_idx = permutation[INITIAL_SIZE:]

X_labeled = X_train[initial_idx]
y_labeled = y_train[initial_idx]
X_pool = X_train[pool_idx]
pool_ids = train_ids[pool_idx]

class SimulatedLabelOracle:
    """只模拟公开 ESOL 的既有标签返回，不代表真实实验。"""
    def __init__(self, labels):
        self._labels = np.asarray(labels)

    def reveal(self, positions):
        return self._labels[np.asarray(positions)]

def make_member(seed, n_estimators=120):
    return RandomForestRegressor(
        n_estimators=n_estimators,
        max_features="sqrt",
        random_state=seed,
        n_jobs=1,
    )

def fit_ensemble(X_fit, y_fit):
    return [
        make_member(seed).fit(X_fit, y_fit)
        for seed in MEMBER_SEEDS
    ]

def ensemble_predict(models, X_part):
    matrix = np.vstack([model.predict(X_part) for model in models])
    return matrix.mean(axis=0), matrix.std(axis=0, ddof=0)

## Steps

### 1. Query 前只使用已标注数据和 `X_pool`

此 cell 不读取 oracle 标签。并列分数使用候选 ID 作为确定性次排序键。

In [3]:
initial_models = fit_ensemble(X_labeled, y_labeled)
pool_prediction_matrix = np.vstack([
    model.predict(X_pool) for model in initial_models
])
pool_mean = pool_prediction_matrix.mean(axis=0)
acquisition_score = pool_prediction_matrix.std(axis=0, ddof=0)

order = np.lexsort((pool_ids, -acquisition_score))
active_positions = order[:BATCH_SIZE]
active_query = pd.DataFrame({
    "candidate_id": pool_ids[active_positions],
    "predicted_logS": pool_mean[active_positions],
    "acquisition_score": acquisition_score[active_positions],
    "reason": "top ensemble disagreement",
})

# 随机对照也必须在揭示任何 pool 标签前固定。
random_rng = np.random.default_rng(RANDOM_QUERY_SEED)
random_positions = random_rng.choice(
    len(X_pool), size=BATCH_SIZE, replace=False
)
display(active_query.round(4))

,candidate_id,predicted_logS,acquisition_score,reason
0,CCCCCC=O,-2.9585,0.3419,top ensemble disagreement
1,CCCCCCCCl,-5.5518,0.3084,top ensemble disagreement
2,CCCCCCCl,-5.5518,0.3084,top ensemble disagreement
3,CCCC(C)CO,-1.6539,0.2994,top ensemble disagreement
4,Clc1cc(c(Cl)c(Cl)c1Cl)c2cc(Cl)c(Cl)c(Cl)c2Cl,-4.4400,0.2726,top ensemble disagreement
5,CCOc1ccc(NC(=O)C)cc1,-3.1297,0.2656,top ensemble disagreement
6,c1c(NC(=O)OC(C)C(=O)NCC)cccc1,-3.0970,0.2574,top ensemble disagreement
7,CC(=O)Nc1ccc(O)cc1,-2.5654,0.2556,top ensemble disagreement
8,Oc1c(Br)cc(C#N)cc1Br,-2.4902,0.2554,top ensemble disagreement
9,Cn1ccc(=O)[nH]c1=O,-2.7142,0.2549,top ensemble disagreement


### 2. Query 固定后才模拟标签返回

下面是权限分界线。返回的仍是公开 ESOL 已有标签，不是真实实验。

In [4]:
# ===== 主动与随机 QUERY 均已固定；以下才模拟实验标签返回 =====
oracle = SimulatedLabelOracle(y_train[pool_idx])
active_y = oracle.reveal(active_positions)
X_active_next = np.concatenate(
    [X_labeled, X_pool[active_positions]], axis=0
)
y_active_next = np.concatenate([y_labeled, active_y], axis=0)
random_y = oracle.reveal(random_positions)
X_random_next = np.concatenate(
    [X_labeled, X_pool[random_positions]], axis=0
)
y_random_next = np.concatenate([y_labeled, random_y], axis=0)

### 3. 在同一固定外部 valid 比较

三行使用相同模型集合和对照集；valid 在课程中已被查看过，因此这不是无偏最终性能。一次模拟不用于宣布策略优越。

In [5]:
scenarios = {
    "before_query": (X_labeled, y_labeled),
    "after_disagreement_query": (X_active_next, y_active_next),
    "after_random_query": (X_random_next, y_random_next),
}
rows = []
for name, (X_fit, y_fit) in scenarios.items():
    models = fit_ensemble(X_fit, y_fit)
    valid_mean, _ = ensemble_predict(models, X_valid)
    rows.append({
        "scenario": name,
        "labeled_size": len(y_fit),
        "valid_rmse": root_mean_squared_error(y_valid, valid_mean),
    })
simulation_results = pd.DataFrame(rows)
display(simulation_results.round(4))

,scenario,labeled_size,valid_rmse
0,before_query,160,1.9416
1,after_disagreement_query,170,1.9343
2,after_random_query,170,1.9057


## Checks

检查区域互斥、预算、query 唯一性和两种更新后的样本量。

In [6]:
assert set(initial_idx).isdisjoint(set(pool_idx))
assert len(initial_idx) + len(pool_idx) == len(X_train)
assert len(active_query) == BATCH_SIZE
assert active_query["candidate_id"].is_unique
assert len(y_active_next) == len(y_random_next) == INITIAL_SIZE + BATCH_SIZE
assert simulation_results["scenario"].is_unique
assert np.isfinite(simulation_results["valid_rmse"]).all()
print("Pool-simulation checks passed. No real experiment occurred.")
print("This one-round table is not evidence that one strategy is superior.")

Pool-simulation checks passed. No real experiment occurred.
This one-round table is not evidence that one strategy is superior.


## Next Steps

本教程只完成单轮机制。正式比较要对多个预注册初始化种子运行多轮预算并保存完整学习曲线。真实项目还需候选配方字段、权限记录和化学可行性审核，算法排名不能自动转成实验任务。